<a href="https://colab.research.google.com/github/weagan/LAYA-JEV/blob/main/laya_implementation_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT-to-LAYA Structural Decision Adaptation (Fixed Layer Indexing)
This notebook resolves the `IndexError` by correctly indexing the first item `[0]` of the batch sequence vector inside the layer weight tracing logic. It evaluates the model **before training**, runs an expanded training loop, and validates post-tuning outcomes.

In [1]:
# 1. SETUP CORE ENVIRONMENT & WARNING FILTERING
import os
import sys
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW

warnings.filterwarnings("ignore", message=".*Error while fetching `HF_TOKEN`.*")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
elif 'kaggle_secrets' in sys.modules:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
else:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [2]:
# 2. DESIGN THE INTEGRATED LAYA ATTENTION MULTI-HEAD ARCHITECTURE
class BertLayaModel(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_choices=4, num_scores=5):
        super(BertLayaModel, self).__init__()
        self.bert = BertModel.from_pretrained(model_name, output_hidden_states=True)
        self.hidden_dim = self.bert.config.hidden_size

        self.layer_attention = nn.Linear(self.hidden_dim, 1)

        self.choice_head = nn.Linear(self.hidden_dim, num_choices)
        self.score_head = nn.Linear(self.hidden_dim, num_scores)
        self.noul_head = nn.Linear(self.hidden_dim, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        cls_embeddings = torch.stack([layer[:, 0, :] for layer in hidden_states], dim=1) # [Batch, 13, Hidden]

        attn_logits = self.layer_attention(cls_embeddings).squeeze(-1) # [Batch, 13]
        attn_weights = torch.softmax(attn_logits, dim=-1).unsqueeze(-1) # [Batch, 13, 1]

        aggregated_representation = torch.sum(cls_embeddings * attn_weights, dim=1) # [Batch, Hidden]

        choice_logits = self.choice_head(aggregated_representation)
        score_logits = self.score_head(aggregated_representation)
        noul_prob = torch.sigmoid(self.noul_head(aggregated_representation))

        return {
            "choice": choice_logits,
            "score": score_logits,
            "noul": noul_prob,
            "layer_weights": attn_weights.squeeze(-1) # [Batch, 13]
        }

In [3]:
# 3. CONSTRUCT EXPANDED SYNTHETIC TRAINING DATASET
CHOICE_OPTIONS = ["billing", "technical", "marketing", "general_inquiry"]
SCORE_OPTIONS = ["very_low", "low", "medium", "high", "critical"]

class SyntheticLayaDataset(Dataset):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        # Expanded diverse dataset matching the structural outputs
        self.data = [
            # Billing related inputs
            {"text": "Charged twice for the subscription link invoice #44.", "c": 0, "s": 4, "n": 0.0},
            {"text": "Requesting a refund for an accidental renewal transaction on my account.", "c": 0, "s": 3, "n": 0.0},
            {"text": "Where can I find my monthly billing updates statement download link?", "c": 0, "s": 1, "n": 0.0},
            # Technical related inputs
            {"text": "Login portal gives an immediate 500 error sequence block.", "c": 1, "s": 3, "n": 0.0},
            {"text": "The database API connection times out when fetching heavy logs.", "c": 1, "s": 4, "n": 0.0},
            {"text": "How do I integrate the webhook callback inside my project environment?", "c": 1, "s": 2, "n": 0.0},
            # Marketing & Spam related inputs
            {"text": "ALERT: Crypto coin drops now. Free bonus access code inside.", "c": 2, "s": 1, "n": 1.0},
            {"text": "Get cheap rates on insurance plans by clicking this promotional link!", "c": 2, "s": 0, "n": 1.0},
            {"text": "Congratulations, your profile was selected to win a luxury prize card.", "c": 2, "s": 0, "n": 1.0},
            # General inquiry inputs
            {"text": "What time does your corporate support platform reopen?", "c": 3, "s": 2, "n": 0.0},
            {"text": "Can you provide a list of your platform documentation links?", "c": 3, "s": 1, "n": 0.0},
            {"text": "I would like to schedule an introductory sales meeting with your team.", "c": 3, "s": 2, "n": 0.0}
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = f"[CONTEXT]: {item['text']} [QUESTIONS]: Triage decision variables."
        encoding = self.tokenizer(prompt, max_length=128, padding='max_length', truncation=True, return_tensors="pt")

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'choice_label': torch.tensor(item['c'], dtype=torch.long),
            'score_label': torch.tensor(item['s'], dtype=torch.long),
            'noul_label': torch.tensor(item['n'], dtype=torch.float)
        }

In [4]:
# 4. INITIALIZE MODEL & DECLARE REUSABLE INFERENCE FUNCTION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertLayaModel(num_choices=len(CHOICE_OPTIONS), num_scores=len(SCORE_OPTIONS)).to(device)

def run_inference_test(stage_label):
    model.eval()
    test_phrase = "Urgent: Payment failed twice but money left my bank account directly."
    formatted_test = f"[CONTEXT]: {test_phrase} [QUESTIONS]: Routing analysis indices status?"

    with torch.no_grad():
        inputs = tokenizer(formatted_test, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model(inputs['input_ids'], inputs['attention_mask'])

        pred_c = torch.argmax(outputs['choice'], dim=-1).item()
        pred_s = torch.argmax(outputs['score'], dim=-1).item()
        pred_n = outputs['noul'].item()

        # FIX: Extract row index 0 out of the shape [1, 13] matrix to capture the sequence vector correctly
        layer_weights_vector = outputs['layer_weights'][0]
        top_layer = torch.argmax(layer_weights_vector).item()

        print(f"\n=== LAYA OUTCOMES ({stage_label}) ===")
        print(f"Input Prompt: {test_phrase}")
        print(f"-> Predicted Category (Choice Head)  : {CHOICE_OPTIONS[pred_c]}")
        print(f"-> Predicted Urgency (Score Head)     : {SCORE_OPTIONS[pred_s]}")
        print(f"-> Spam Probability (Noul Head)      : {pred_n:.4f}")
        print(f"-> Dominant Attention Layer Focus    : Layer {top_layer} (Value Coefficient: {layer_weights_vector[top_layer].item():.4f})")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# 5. TEST RUN BEFORE TRAINING (UNTRAINED BASELINE VALIDATION)
print("Running validation check on completely untrained / random heads initialization weights...")
run_inference_test("BEFORE FINE-TUNING")

Running validation check on completely untrained / random heads initialization weights...

=== LAYA OUTCOMES (BEFORE FINE-TUNING) ===
Input Prompt: Urgent: Payment failed twice but money left my bank account directly.
-> Predicted Category (Choice Head)  : technical
-> Predicted Urgency (Score Head)     : critical
-> Spam Probability (Noul Head)      : 0.5013
-> Dominant Attention Layer Focus    : Layer 12 (Value Coefficient: 0.1044)


In [6]:
# 6. EXECUTE MULTI-TASK FINE-TUNING PIPELINE
dataset = SyntheticLayaDataset(tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)

print("\nBeginning Model Fine-Tuning Sequence on Expanded Dataset...")
model.train()
for epoch in range(4): # Enhanced calibration tracking over 4 iterations
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask)

        loss_c = F.cross_entropy(outputs['choice'], batch['choice_label'].to(device))
        loss_s = F.cross_entropy(outputs['score'], batch['score_label'].to(device))
        loss_n = F.binary_cross_entropy(outputs['noul'].squeeze(-1), batch['noul_label'].to(device))

        combined_loss = loss_c + loss_s + loss_n
        combined_loss.backward()
        optimizer.step()
        total_loss += combined_loss.item()

    print(f"Epoch {epoch + 1}/4 Complete. Combined Training Loss Matrix: {total_loss:.4f}")


Beginning Model Fine-Tuning Sequence on Expanded Dataset...
Epoch 1/4 Complete. Combined Training Loss Matrix: 22.5011
Epoch 2/4 Complete. Combined Training Loss Matrix: 21.3633
Epoch 3/4 Complete. Combined Training Loss Matrix: 20.8434
Epoch 4/4 Complete. Combined Training Loss Matrix: 19.8873


In [7]:
# 7. POST-TRAINING INFERENCE TESTING FRAMEWORK (POST-TUNING RESULTS)
print("\nRunning evaluation check on fine-tuned weights and adapted attention projections...")
run_inference_test("AFTER FINE-TUNING")


Running evaluation check on fine-tuned weights and adapted attention projections...

=== LAYA OUTCOMES (AFTER FINE-TUNING) ===
Input Prompt: Urgent: Payment failed twice but money left my bank account directly.
-> Predicted Category (Choice Head)  : technical
-> Predicted Urgency (Score Head)     : low
-> Spam Probability (Noul Head)      : 0.2190
-> Dominant Attention Layer Focus    : Layer 12 (Value Coefficient: 0.1895)
